In [1]:
language = 'pt'

### 1. Gravação de Áudio com Python (e Um Pouco de JavaScript) 🗣

In [3]:
from IPython.display import Audio, display, Javascript
from google.colab import output
from base64 import b64decode

RECORD = """
const sleep = time => new Promise(resolve => setTimeout(resolve, time))
const b2text = blob => new Promise(resolve => {
  const reader = new FileReader()
  reader.onloadend = e => resolve(e.srcElement.result)
  reader.readAsDataURL(blob)
})
var record = time => new Promise(async resolve => {
  stream = await navigator.mediaDevices.getUserMedia({ audio: true })
  recorder = new MediaRecorder(stream)
  chunks = []
  recorder.ondataavailable = e => chunks.push(e.data)
  recorder.start()
  await sleep(time)
  recorder.onstop = async ()=>{
    blob = new Blob(chunks)
    text = await b2text(blob)
    resolve(text)
  }
  recorder.stop()
})"""


def record(sec=5):
  display(Javascript(RECORD))
  js_result = output.eval_js('record(%s)' % (sec * 1000))
  audio = b64decode(js_result.split(',')[1])
  file_name = 'request_audio.wav'
  with open(file_name, 'wb') as f:
    f.write(audio)
  return f'/content/{file_name}'

print('Gravação de Áudio Iniciada...\n')
record_file = record()
display(Audio(record_file, autoplay=True))

Gravação de Áudio Iniciada...



<IPython.core.display.Javascript object>

### 2. Reconhecimento de Fala com Whisper (OpenAI) 🧠

In [4]:
!pip install git+https://github.com/openai/whisper.git -q

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.3/188.3 MB 6.2 MB/s eta 0:00:00


In [5]:
import whisper

model = whisper.load_model("small")
result = model.transcribe(record_file, fp16=False, language=language)
transcription = result["text"]
print(transcription)

100%|███████████████████████████████████████| 461M/461M [00:06<00:00, 76.5MiB/s]


 Estes são um, dois.


### 3. Utilizando Comandos por Voz para Calcular valores ⚙

In [ ]:
import re
from time import sleep

def linha_p(msg):
  print("-" * 30)
  print(msg)
  print("-" * 30)


def linha(msg):
  print(msg)
  print("-" * 30)


def extrair_numeros(texto):
    numeros = re.findall(r'\d+(?:\.\d+)?', texto)
    return [float(n) for n in numeros]


def identificar_operacao(texto):

    texto = texto.lower()

    if "som" in texto or "mais" in texto:
        return "soma"
    elif "sub" in texto or "menos" in texto:
        return "subtracao"
    elif "multip" in texto or "vezes" in texto:
        return "multiplicacao"
    elif "div" in texto:
        return "divisao"
    elif "sair" in texto:
        return "sair"
    else:
        return None


linha_p("Bem-vindo(a) à Calculadora por Voz!")
linha("Para usar, basta dizer a operação e em \nseguida os dois números que deseja usar! \nExemplo: Somar 10 e 2")

while True:

    sleep(0.5)
    linha_p("Gravação iniciada...")
    print("Fale algo!")
    record_file = record(sec=5)
    linha("Transcevendo...")

    result = model.transcribe(record_file, fp16=False, language=language)
    comando = result["text"]

    print("Você disse:", comando)

    operacao = identificar_operacao(comando)

    if operacao == "sair":
        print("Encerrando calculadora.")
        break

    numeros = extrair_numeros(comando)

    if len(numeros) < 2:
        linha("Não consegui identificar dois números.")
        continue

    a, b = numeros[0], numeros[1]

    if operacao == "soma":
        print("Resultado:", a + b)
    elif operacao == "subtracao":
        print("Resultado:", a - b)
    elif operacao == "multiplicacao":
        print("Resultado:", a * b)
    elif operacao == "divisao":
        if b == 0:
            print("Divisão por zero não é permitida neste universo.")
        else:
            print("Resultado:", a / b)
    else:
        print("Operação não reconhecida.")



------------------------------
Bem-vindo(a) à Calculadora por Voz!
------------------------------
Para usar, basta dizer a operação e em 
seguida os dois números que deseja usar! 
Exemplo: Somar 10 e 2
------------------------------
Gravação iniciada...
Fale algo!
------------------------------


<IPython.core.display.Javascript object>

Transcevendo...
------------------------------
Você disse:  Dividir 4 e 2.
Resultado: 2.0
Gravação iniciada...
Fale algo!
------------------------------


<IPython.core.display.Javascript object>

Transcevendo...
------------------------------
Você disse:  Somar 1 e 2.
Resultado: 3.0
Gravação iniciada...
Fale algo!
------------------------------


<IPython.core.display.Javascript object>

KeyboardInterrupt: 